# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Choose an appropriate scaler based on feature range, outliers, sparsity, and estimator sensitivity. 
- Apply distribution transformations to reduce skewness or map features to target distributions. 
- Compare transformed feature distributions visually and numerically before modeling. 


## **1. Scaling Methods**

### **1.1. Selecting a Scaler**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_01.jpg?v=1783831871" width="250">



>* Compare feature meanings and numeric ranges first.
>* Scaling prevents large units dominating model behavior.

>* Outliers can distort many scaling methods.
>* Preserve sparsity when zeros carry meaning.

>* Scale matters for distance-based models.
>* Match scaler to data and estimator.



### **1.2. Choosing the Right Scaler**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_02.jpg?v=1783831873" width="250">



>* Feature ranges affect model behavior significantly.
>* Choose scaling by bounds and spread.

>* Outliers can distort scaling and compress data.
>* Choose scalers that preserve sparsity constraints.

>* Scaling importance depends on the algorithm.
>* Match scaler to data and model.



In [ ]:
#@title Python Code - Choosing the Right Scaler

# This script compares common scaling choices.
# It highlights range, outliers, and sparsity.
# One plot summarizes when each scaler fits.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Build three contrasting features.
age_years = np.array([18, 25, 32, 40, 52, 61], dtype=float)
income_usd = np.array([32000, 45000, 52000, 61000, 72000, 500000], dtype=float)
click_counts = np.array([0, 0, 3, 0, 8, 0], dtype=float)

# Define simple scaling helpers.
def minmax_scale(x):
    return (x - x.min()) / (x.max() - x.min())

def zscore_scale(x):
    return (x - x.mean()) / x.std(ddof=0)

# Use median and IQR.
def robust_scale(x):
    q1, q3 = np.percentile(x, [25, 75])
    return (x - np.median(x)) / (q3 - q1)

# Apply each scaler.
scaled = pd.DataFrame({
    "Age MinMax": minmax_scale(age_years),
    "Income Standard": zscore_scale(income_usd),
    "Income Robust": robust_scale(income_usd),
    "Clicks MaxAbs": click_counts / np.abs(click_counts).max(),
})

# Check tiny shapes safely.
assert scaled.shape == (6, 4), "Unexpected transformed shape."

# Print a short decision guide.
print("Age range is moderate: MinMax keeps values between 0 and 1.")
print("Income has an outlier: Robust scaling resists the $500000 value.")
print("Sparse click counts include many zeros: MaxAbs keeps zeros unchanged.")
print("Distance and gradient models usually need scaling more than trees.")
print("Standard scaling can be fine when outliers are limited.")
print("Below, compare standard versus robust income scaling visually.")

# Plot transformed values for comparison.
ax = scaled[["Income Standard", "Income Robust"]].plot(
    kind="bar", figsize=(8, 4), rot=0
)

# Add clear labels.
ax.set_title("Income Feature: Standard vs Robust Scaling")
ax.set_xlabel("Sample index")
ax.set_ylabel("Scaled value")
plt.tight_layout()
plt.show()



### **1.3. Robust Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_03.jpg?v=1783831875" width="250">



>* Best when outliers distort standard scaling.
>* Uses median and IQR to resist extremes.

>* Outliers can distort scale-sensitive learning algorithms.
>* Robust scaling preserves typical variation better.

>* Use robust scaling when outliers matter less.
>* Avoid it for skewed or sparse data.



## **2. Sparse Safe Scaling**

### **2.1. Preserving Sparse Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_01.jpg?v=1783831859" width="250">



>* Keep zeros unchanged in sparse data.
>* Transform nonzeros without creating false signals.

>* Transform only nonzero values, keep zeros unchanged.
>* Compress extremes to reduce skew safely.

>* Keeping zeros preserves meaning and interpretability.
>* Sparse-safe transforms improve skew without losing sparsity.



In [ ]:
#@title Python Code - Preserving Sparse Structure

# Sparse transforms should keep zeros unchanged.
# This example compares safe and unsafe scaling.
# We inspect sparsity and skewness visually.

import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt

# Build a tiny sparse count matrix.
counts = np.array([0, 0, 1, 2, 8, 30])
X_sparse = sparse.csr_matrix(counts.reshape(-1, 1))

# Confirm the original sparse pattern.
original_dense = X_sparse.toarray().ravel()
original_zeros = np.sum(original_dense == 0)

# Apply a sparse-safe log transform.
safe_dense = np.log1p(original_dense)
safe_zeros = np.sum(safe_dense == 0)

# Show an unsafe centering transform.
unsafe_dense = original_dense - original_dense.mean()
unsafe_zeros = np.sum(unsafe_dense == 0)

# Print a short numerical comparison.
print(f"Original zeros: {original_zeros}")
print(f"After log1p zeros: {safe_zeros}")
print(f"After centering zeros: {unsafe_zeros}")
print(f"Original nonzeros: {original_dense[original_dense > 0].tolist()}")
print(f"log1p nonzeros: {np.round(safe_dense[safe_dense > 0], 2).tolist()}")

# Plot distributions for the three versions.
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].bar(range(len(original_dense)), original_dense, color="steelblue")
axes[1].bar(range(len(safe_dense)), safe_dense, color="seagreen")
axes[2].bar(range(len(unsafe_dense)), unsafe_dense, color="tomato")

# Add clear titles and labels.
axes[0].set_title("Original counts")
axes[1].set_title("Sparse-safe log1p")
axes[2].set_title("Unsafe centering")
axes[0].set_ylabel("Value")

# Keep the layout compact.
for ax in axes:
    ax.set_xlabel("Sample")

plt.tight_layout()
plt.show()



### **2.2. Preserving Sparse Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_02.jpg?v=1783831861" width="250">



>* Zeros in sparse data carry meaning.
>* Use transforms that keep zeros unchanged.

>* Transform only nonzeros; keep zeros unchanged.
>* Compress large active values without creating activity.

>* Zeros preserve meaning and feature comparability.
>* Sparse-safe transforms aid interpretation and efficiency.



In [ ]:
#@title Python Code - Preserving Sparse Structure

# Sparse data often contains meaningful zeros.
# Good transformations should preserve zero locations.
# This example compares sparse safe choices.

# Import compact tools for arrays and plotting.
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt

# Set a deterministic seed for reproducibility.
rng = np.random.default_rng(7)

# Build sparse like count data with many zeros.
counts = np.zeros(120, dtype=float)
active = rng.choice(120, size=18, replace=False)
counts[active] = rng.integers(1, 80, size=18)

# Create a sparse matrix representation.
X_sparse = sparse.csr_matrix(counts.reshape(-1, 1))

# Apply a sparse safe log transform.
log_safe = np.log1p(counts)

# Apply mean centering that breaks sparsity.
centered = counts - counts.mean()

# Count nonzero entries before and after.
original_nonzero = np.count_nonzero(counts)
log_nonzero = np.count_nonzero(log_safe)
centered_nonzero = np.count_nonzero(centered)

# Summarize skewness with a simple helper.
def skewness(values):
    mean = values.mean()
    std = values.std()

    if std == 0:
        return 0.0
    z = (values - mean) / std
    return float(np.mean(z ** 3))

# Print a short numerical comparison.
print(f"Original nonzeros: {original_nonzero}")
print(f"log1p nonzeros: {log_nonzero}")
print(f"Centered nonzeros: {centered_nonzero}")
print(f"Original skewness: {skewness(counts):.2f}")
print(f"log1p skewness: {skewness(log_safe):.2f}")
print(f"Sparse matrix stored values: {X_sparse.nnz}")

# Plot distributions for the active values only.
active_original = counts[counts > 0]
active_log = log_safe[log_safe > 0]

# Create one compact comparison figure.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(active_original, bins=8, color="steelblue", edgecolor="black")
axes[0].set_title("Nonzero counts")
axes[0].set_xlabel("Original values")

axes[1].hist(active_log, bins=8, color="darkorange", edgecolor="black")
axes[1].set_title("After log1p")
axes[1].set_xlabel("Transformed values")

# Finish the figure neatly.
plt.tight_layout()
plt.show()



### **2.3. Sparse Aware Transforms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_03.jpg?v=1783831863" width="250">



>* Zeros often encode meaningful absence patterns.
>* Transforms should reduce skew without changing zeros.

>* Compress large positives while keeping zeros unchanged.
>* Choose transforms that preserve sparsity and meaning.

>* Reduce skewness while keeping zeros unchanged.
>* Preserve sparsity for efficient, scalable modeling.



In [ ]:
#@title Python Code - Sparse Aware Transforms

# Sparse transforms can preserve meaningful zero entries.
# This example compares raw and log transformed counts.
# Zeros stay zero while large values compress.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Build a tiny sparse count feature.
counts = np.array(
    [0, 0, 0, 1, 2, 3, 8, 20, 60, 150],
    dtype=float,
)

# Apply a sparse safe logarithmic transform.
log_counts = np.log1p(counts)
zero_preserved = bool(
    np.all(log_counts[counts == 0] == 0)
)

# Compute simple skewness without extra libraries.
def skewness(values):
    centered = values - values.mean()
    scale = values.std(ddof=0)

    if scale == 0:
        return 0.0
    return np.mean((centered / scale) ** 3)

# Summarize the transformation numerically.
raw_skew = skewness(counts)
log_skew = skewness(log_counts)
nonzero_before = int(np.count_nonzero(counts))
nonzero_after = int(np.count_nonzero(log_counts))

# Print a short teaching summary.
print(f"Zeros preserved: {zero_preserved}")
print(f"Nonzero entries before: {nonzero_before}")
print(f"Nonzero entries after: {nonzero_after}")
print(f"Raw skewness: {raw_skew:.2f}")

print(f"Log1p skewness: {log_skew:.2f}")
print(f"Largest raw value: {counts.max():.0f}")
print(f"Largest transformed value: {log_counts.max():.2f}")

# Plot both distributions for comparison.
fig, axes = plt.subplots(
    1, 2, figsize=(10, 4)
)
sns.histplot(counts, bins=8, ax=axes[0], color="steelblue")
axes[0].set_title("Original sparse counts")

sns.histplot(log_counts, bins=8, ax=axes[1], color="darkorange")
axes[1].set_title("After log1p transform")
axes[1].set_xlabel("Transformed value")

plt.tight_layout()
plt.show()



## **3. Distribution Comparison Tools**

### **3.1. Visual Distribution Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_01.jpg?v=1783831865" width="250">



>* Plots quickly reveal hidden distribution changes.
>* Transformations should reduce distortion, preserve meaning.

>* Check spread, tails, spikes, and clumping.
>* Ensure transformations preserve meaningful data patterns.

>* Visuals compare how transformations reshape distributions.
>* They reveal effects before numerical evaluation.



In [ ]:
#@title Python Code - Visual Distribution Checks

# Visual checks reveal transformation effects quickly.
# This example compares skewed feature distributions.
# One figure and short summaries are shown.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a small skewed feature.
income = rng.lognormal(mean=10.0, sigma=0.9, size=400)

# Add a few larger outliers.
income[:6] = income[:6] * 6

# Store values in a dataframe.
df = pd.DataFrame({"original": income})

# Apply a log style transformation.
df["log1p"] = np.log1p(df["original"])

# Apply a power transformation.
power = PowerTransformer(method="yeo-johnson", standardize=False)

df["yeo_johnson"] = power.fit_transform(df[["original"]]).ravel()

# Apply a quantile normal mapping.
quantile = QuantileTransformer(
    output_distribution="normal", n_quantiles=100, random_state=7
)

df["quantile_normal"] = quantile.fit_transform(df[["original"]]).ravel()

# Check that all columns align.
assert df.shape == (400, 4)

# Summarize skewness before plotting.
skew_values = df.skew(numeric_only=True).round(2)

# Summarize spread using interquartile range.
iqr_values = (df.quantile(0.75) - df.quantile(0.25)).round(2)

# Print short numeric comparisons.
print("Skewness by version:")
print(skew_values.to_string())
print("\nIQR by version:")
print(iqr_values.to_string())

# Build one figure with comparisons.
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

# Pair names with subplot axes.
pairs = list(zip(df.columns, axes.ravel()))

# Draw each histogram consistently.
for name, ax in pairs:
    ax.hist(df[name], bins=25, color="steelblue", edgecolor="white")

    ax.set_title(name.replace("_", " ").title())
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

# Improve spacing for readability.
plt.tight_layout()
plt.show()



### **3.2. Visual Distribution Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_02.jpg?v=1783831867" width="250">



>* Plots quickly reveal distribution changes after transformation.
>* Visual checks confirm skew reduction and balance.

>* Check fit for later modeling needs.
>* Spot distortion, lost groups, or noise.

>* Compare multiple transformations, not just before-after.
>* Visual checks confirm feature-specific preprocessing quality.



In [ ]:
#@title Python Code - Visual Distribution Checks

# Visual checks reveal distribution changes clearly.
# This example compares original and transformed values.
# One plot supports quick preprocessing decisions.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)
# Create a small right skewed feature.
visits = rng.lognormal(mean=2.0, sigma=0.9, size=300)

# Apply a simple skew reducing transform.
log_visits = np.log1p(visits)
# Compute compact numeric summaries.
orig_skew = ((visits - visits.mean()) ** 3).mean() / visits.std() ** 3
trans_skew = ((log_visits - log_visits.mean()) ** 3).mean() / log_visits.std() ** 3

# Print short before and after summaries.
print(f"Original mean: {visits.mean():.2f}")
print(f"Original skew: {orig_skew:.2f}")
print(f"Transformed mean: {log_visits.mean():.2f}")
print(f"Transformed skew: {trans_skew:.2f}")

# Build side by side histograms.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(visits, bins=25, color="steelblue", edgecolor="black")
axes[0].set_title("Original feature")
axes[0].set_xlabel("Daily website visits")

axes[1].hist(log_visits, bins=25, color="seagreen", edgecolor="black")
axes[1].set_title("After log1p transform")
axes[1].set_xlabel("Transformed visits")

# Add one shared teaching title.
fig.suptitle("Visual Distribution Checks")
plt.tight_layout()
plt.show()



### **3.3. Visual Numeric Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_03.jpg?v=1783831869" width="250">



>* Numeric summaries verify distribution changes consistently.
>* Check spread, percentiles, and skewness improvement.

>* Compare consistent statistics for meaningful transformation changes.
>* Check tails, quantiles, and preserved variation.

>* Compare numbers for real modeling impact.
>* Check range, outliers, and useful separation.



In [ ]:
#@title Python Code - Visual Numeric Comparison

# Compare distributions with simple numeric summaries.
# Visual checks become stronger with measurements.
# This example contrasts original and transformed values.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a right skewed feature.
income = rng.lognormal(mean=10.0, sigma=0.8, size=400)

# Transform values using logarithms.
log_income = np.log1p(income)

# Store both versions together.
df = pd.DataFrame({"Original": income, "Log transformed": log_income})

# Define a compact summary function.
def summarize(values, name):
    series = pd.Series(values)
    return pd.Series(
        {
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(),
            "p90": series.quantile(0.90),
            "p99": series.quantile(0.99),
            "skew": series.skew(),
        },
        name=name,
    )

# Check the data sizes first.
assert df.shape == (400, 2)

# Build a side by side summary.
summary = pd.concat(
    [
        summarize(df["Original"], "Original"),
        summarize(df["Log transformed"], "Log transformed"),
    ],
    axis=1,
).round(2)

# Print a short numeric comparison.
print("Numeric comparison before modeling")
print(summary)
print()
print("Smaller skew and tighter upper percentiles suggest")
print("a more balanced distribution after transformation.")

# Plot both distributions for visual comparison.
fig, axes = plt.subplots(ncols=2, figsize=(10, 4))

# Draw the original distribution.
axes[0].hist(df["Original"], bins=30, color="steelblue", edgecolor="black")
axes[0].set_title("Original")
axes[0].set_xlabel("Income")
axes[0].set_ylabel("Count")

# Draw the transformed distribution.
axes[1].hist(df["Log transformed"], bins=30, color="seagreen", edgecolor="black")
axes[1].set_title("Log transformed")
axes[1].set_xlabel("log1p(Income)")
axes[1].set_ylabel("Count")

# Finish the figure neatly.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>


In this lecture, you learned to:
- Choose an appropriate scaler based on feature range, outliers, sparsity, and estimator sensitivity. 
- Apply distribution transformations to reduce skewness or map features to target distributions. 
- Compare transformed feature distributions visually and numerically before modeling. 

In the next Lecture (Lecture B), we will go over 'Feature Expansion'